# Lesson 02: Ingest, from PDF to chunks (solution)

**Goal:** turn a PDF into typed blocks and then into table-preserving chunks, using the real `docrag` code.

**Before you start**
- Theory: [lesson 02](../../../docs/lessons/02_ingest.md).
- Python concepts: generators (`yield`), `itertools.groupby`, `next()` with a generator expression.
- Files used: `fixtures/tiny.pdf` (3 pages: intro, a revenue table, a conclusion). Offline, no API key.

Each step: read the short explanation, fill in the `# TODO` cell, run the check cell below it.

## Setup (just run it)

In [ ]:
import inspect
import itertools
from collections.abc import Iterator
from pathlib import Path

from docrag.ingest.chunker import ChunkerOptions, chunk_blocks
from docrag.ingest.parser import get_parser
from docrag.shared.schema import Block

# Find the repo root (the folder with pyproject.toml), wherever Jupyter started.
ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()
)
pdf_bytes = (ROOT / "fixtures" / "tiny.pdf").read_bytes()
print(f"tiny.pdf: {len(pdf_bytes):,} bytes")

## Step 1: Parse the PDF into blocks

A **parser** turns PDF bytes into a list of `Block`s: one typed piece of a page (`text`, `table` or `image`), with its page number and heading chain already fixed. `get_parser()` returns the default parser (PyMuPDF), and its `.parse(pdf_bytes, doc_id)` does the work.

> `Parser` is a `typing.Protocol`: any class with `name`, `version` and `parse()` counts as a parser, like a TS interface (no `implements` needed).

In [ ]:
blocks = get_parser().parse(pdf_bytes, doc_id="tiny")

In [ ]:
assert blocks is not None, "Hint: blocks = get_parser().parse(...)"
assert len(blocks) == 4, f"Expected 4 blocks in tiny.pdf, got {len(blocks)}"
assert {b.type for b in blocks} == {"text", "table"}, (
    "tiny.pdf has text blocks and one table"
)
assert {b.page for b in blocks} == {1, 2, 3}, (
    "Pages are 1-based and fixed by the parser"
)
for b in blocks:
    print(f"p{b.page} {b.type:5} {' > '.join(b.section_path)}")
print("OK")

## Step 2: A generator that walks the document page by page

A function with `yield` is a **generator**: calling it returns a lazy iterator, and the body runs only as you iterate (TS `function*`, a Rust `Iterator`, a Java `Stream` without `collect`). `itertools.groupby(items, key=...)` groups *consecutive* items with the same key, which works here because blocks come in reading order.

Write `blocks_by_page` so it yields `(page, [blocks on that page])` tuples.

In [ ]:
def blocks_by_page(blocks: list[Block]) -> Iterator[tuple[int, list[Block]]]:
    for page, group in itertools.groupby(blocks, key=lambda b: b.page):
        yield page, list(group)

In [ ]:
pages = blocks_by_page(blocks)
assert inspect.isgenerator(pages), "Hint: the function must `yield`, not return a list"
pages = list(pages)
assert [p for p, _ in pages] == [1, 2, 3], (
    f"Expected pages [1, 2, 3], got {[p for p, _ in pages]}"
)
assert [len(bs) for _, bs in pages] == [1, 2, 1], "Page 2 holds the text AND the table"
for page, bs in pages:
    print(page, [b.type for b in bs])
print("OK")

## Step 3: Chunk, and find the table

`chunk_blocks(blocks)` turns blocks into `Chunk`s, the unit that gets embedded, searched and cited. Rule: **a table is one chunk** (caption + Markdown), never cut mid-row. Note the two texts: `text` stays raw (used to verify quotes), `embed_text` gets the heading chain as a prefix.

Use `next()` with a generator expression to get the first chunk whose `type` is `"table"`. It's like `.find()` in TS.

In [ ]:
chunks = chunk_blocks(blocks)
table = next(c for c in chunks if c.type == "table")

In [ ]:
assert table is not None, "Hint: table = next(c for c in chunks if ...)"
assert table.id == "tiny:p2:c1", f"Chunk IDs read doc:p<page>:c<n>; got {table.id}"
assert "|Europe|41.0|45.5|" in table.text, (
    "The whole Markdown table should be in one chunk"
)
assert table.embed_text.startswith("Northwind Survey Report > 2 Results"), (
    "embed_text gets the heading prefix"
)
assert not table.text.startswith("Northwind"), (
    "text stays raw (no prefix), so quotes can be verified"
)
print(table.text)
print("OK")

## Step 4: A table that is too big for one chunk

Only tables over `max_table_words` get split, **by rows**, and every piece repeats the caption and the header row, so each piece still makes sense on its own in search results. `tiny.pdf`'s table is too small to split, so here is a synthetic 60-row table (the cell is ready, just run it).

In [ ]:
rows = [f"|Region {i}|{i * 1.5:.1f} m EUR|up {i % 7} percent|" for i in range(60)]
caption = "Table 9: Revenue per region"
table_md = "\n".join(["|Region|Revenue|Change|", "|---|---|---|", *rows])
# Built like the parser builds tables: caption separate, table_md = header + rows.
big = Block(
    doc_id="demo",
    type="table",
    page=5,
    section_path=["Annex"],
    order=0,
    caption=caption,
    table_md=table_md,
    text=f"{caption}\n{table_md}",
)
print(len(big.text.split()), "words")

In [ ]:
pieces = [
    c
    for c in chunk_blocks([big], ChunkerOptions(max_table_words=60))
    if c.type == "table"
]

In [ ]:
assert len(pieces) > 1, (
    "With max_table_words=60 the table must be split into several chunks"
)
for p in pieces:
    lines = p.text.splitlines()
    assert lines[:3] == [
        "Table 9: Revenue per region",
        "|Region|Revenue|Change|",
        "|---|---|---|",
    ], "Every piece repeats caption + header row"
body = [line for p in pieces for line in p.text.splitlines()[3:]]
assert body == rows, "Every row appears exactly once, in order: never cut in the middle"
print(f"{len(pieces)} pieces, ids: {[p.id for p in pieces]}")
print("OK")

## Observation question

Run `chunk_blocks(blocks, ChunkerOptions(max_table_words=5))` on **tiny.pdf**. The table has 12 words, more than 5, but it still comes back as **one** chunk. Why?

<details>
<summary>Answer</summary>

Two reasons, both in `_split_table` in `chunker.py`:

1. The budget for rows is `max(max_table_words - words(caption + header), 20)`. The floor of 20 stops a tiny limit from producing one-row chunks that repeat a long header over and over.
2. Words are counted with `str.split()`, and a Markdown row like `|Europe|41.0|45.5|` has no spaces, so it counts as **one** word. Three rows are far below 20.

Splitting only ever happens between rows. A row longer than the budget stays whole and becomes its own piece.
</details>

## The real code

- Parser: [`src/docrag/ingest/parser.py`](../../../src/docrag/ingest/parser.py)
- Chunker: [`src/docrag/ingest/chunker.py`](../../../src/docrag/ingest/chunker.py)
- Parse cache and the full ingest step: [`src/docrag/ingest/pipeline.py`](../../../src/docrag/ingest/pipeline.py), [`src/docrag/ingest/indexer.py`](../../../src/docrag/ingest/indexer.py)
- Theory and interview questions: [`docs/lessons/02_ingest.md`](../../../docs/lessons/02_ingest.md)